# Unidades Deliberativas Verificáveis: de citação exata a validação manual

Uma opinião do LDS é uma string dentro de uma lista, sem nenhum vínculo verificável com a transcrição de onde deveria ter vindo: não dá para responder quem disse aquilo, onde na fala, e com que confiança essa ligação pode ser feita. Este notebook percorre três tentativas de resolver esse problema, cada uma motivada pela limitação da anterior, e termina apontando para onde a próxima tentativa deveria ir.

In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
with open("dataset/PublicHearingBR_LDS.jsonl") as f:
    lds_records = [json.loads(line) for line in f]

sample_hearings = lds_records[:20]

## Tentativa 1: evidência por citação direta

A forma mais simples de ligar uma opinião a uma evidência é procurar, dentro dela, um trecho entre aspas, e tentar localizar esse trecho na própria fala da pessoa. Antes disso, porém, é preciso conseguir separar a fala de cada participante dentro da transcrição.

In [3]:
import re

In [4]:
naive_turn_pattern = re.compile(
    r"(?:O\s+SR\.|A\s+SRA\.)\s*([A-ZÀ-Ü][^(\n]{0,60})\([^)]*\)\s*-\s"
)
sample_transcript = sample_hearings[0]["transcricao"]
naive_matches = naive_turn_pattern.findall(sample_transcript)
print(f"turnos encontrados exigindo parênteses: {len(naive_matches)}")
print("nomes únicos:", sorted({m.strip() for m in naive_matches}))

turnos encontrados exigindo parênteses: 53
nomes únicos: ['ADRIANA VENTURA', 'AFONSO HAMM', 'ARLINDO CHINAGLIA', 'BIA KICIS', 'FILIPE BARROS', 'FLORENTINO NETO', 'GENERAL GIRÃO', 'GILVAN DA FEDERAL', 'LUIZ PHILIPPE DE ORLEANS E BRAGANÇA', 'MARCEL VAN HATTEM', 'MARCELO MORAES', 'MARIO FRIAS', 'PRESIDENTE']


In [5]:
missing_person = "Michael Shellenberger"
position = sample_transcript.upper().find(missing_person.split()[0].upper(), 5000)
print(sample_transcript[position - 20 : position + 60])

laborador americano Michael Shellenberger.

Confesso que eu nem sei ao certo a o


O nome aparece, mas sem parênteses depois: convidados sem filiação partidária (especialistas, testemunhas estrangeiras) têm um cabeçalho de turno diferente. Um regex que exige parênteses parece funcionar, porque encontra turnos em toda audiência, mas exclui esses participantes silenciosamente. A pipeline compartilhada do projeto já corrige isso.

In [6]:
from utils.udv_pipeline import (
    best_semantic_match,
    extract_quote,
    find_quote_evidence,
    resolve_person_speech,
    resolve_turn_name,
    split_into_turns,
    split_sentences,
)

In [7]:
fixed_turns = [
    dict(turn, resolved_name=resolve_turn_name(turn)) for turn in split_into_turns(sample_transcript)
]
print(f"turnos encontrados com o regex corrigido: {len(fixed_turns)}")

turnos encontrados com o regex corrigido: 65


Com a segmentação de turnos corrigida, o próximo passo é resolver, para cada participante listado na metadata, quais turnos são dele, e então tentar achar uma citação direta de cada opinião na fala correspondente.

In [8]:
people_total = 0
people_resolved = 0
quote_only_counts = {"no_quote": 0, "quote_not_found": 0, "quote_found": 0, "person_not_resolved": 0}

for hearing in sample_hearings:
    turns = [dict(t, resolved_name=resolve_turn_name(t)) for t in split_into_turns(hearing["transcricao"])]
    for pessoa in hearing["metadados"]["envolvidos"]:
        people_total += 1
        matched_turns, speech = resolve_person_speech(pessoa, turns)
        if matched_turns:
            people_resolved += 1
        for opinion_text in pessoa["opinioes"]:
            if not matched_turns:
                quote_only_counts["person_not_resolved"] += 1
                continue
            quote = extract_quote(opinion_text)
            if quote is None:
                quote_only_counts["no_quote"] += 1
            elif find_quote_evidence(quote, speech) is not None:
                quote_only_counts["quote_found"] += 1
            else:
                quote_only_counts["quote_not_found"] += 1

In [9]:
print(f"pessoas resolvidas: {people_resolved}/{people_total}")
pd.Series(quote_only_counts, name="opinioes")

pessoas resolvidas: 101/108


no_quote               151
quote_not_found         71
quote_found             32
person_not_resolved     14
Name: opinioes, dtype: int64

Resolver a pessoa por trás de cada turno funciona bem (101 de 108 participantes). Mas ligar cada opinião a uma citação exata funciona mal: das 268 opiniões, 151 nem têm uma citação direta entre aspas (são paráfrase da matéria), 71 têm citação mas ela não foi localizada na fala da pessoa, e só 32 tiveram a citação efetivamente encontrada. Buscar por correspondência exata de texto cobre uma fração pequena do problema, porque a maioria das opiniões é reformulada, não copiada.

## Tentativa 2: similaridade textual (TF-IDF)

Se a maioria das opiniões é paráfrase, o próximo passo natural é comparar significado por vocabulário compartilhado, não por texto idêntico. TF-IDF é exatamente o baseline de comparação textual já previsto para o projeto, não uma escolha arbitrária. Para isso, a fala de cada pessoa é dividida em sentenças, e cada opinião é comparada contra essas sentenças.

In [10]:
sample_speech = resolve_person_speech(
    sample_hearings[0]["metadados"]["envolvidos"][0], fixed_turns
)[1]
sample_sentences = split_sentences(sample_speech)
print(f"sentenças extraídas do primeiro participante: {len(sample_sentences)}")

sentenças extraídas do primeiro participante: 72


A comparação de cada opinião contra as sentenças usa `TfidfVectorizer` e similaridade de cosseno, guardando a sentença de maior similaridade e o escore. Mas um escore de similaridade sozinho não diz o que é uma evidência confiável: para isso, é preciso calibrar um corte.

In [11]:
import random

random.seed(42)

In [12]:
positive_scores = []
negative_scores = []
all_sentences_pool = []

for hearing in sample_hearings:
    turns = [dict(t, resolved_name=resolve_turn_name(t)) for t in split_into_turns(hearing["transcricao"])]
    person_sentences = {}
    for pessoa in hearing["metadados"]["envolvidos"]:
        _, speech = resolve_person_speech(pessoa, turns)
        sentences = split_sentences(speech)
        person_sentences[pessoa["nome"]] = sentences
        all_sentences_pool.extend(sentences)

    for pessoa in hearing["metadados"]["envolvidos"]:
        sentences = person_sentences[pessoa["nome"]]
        if not sentences:
            continue
        for opinion_text in pessoa["opinioes"]:
            quote = extract_quote(opinion_text)
            if quote is None:
                continue
            found = find_quote_evidence(quote, " ".join(sentences))
            if found is None:
                continue
            true_sentence = next((s for s in sentences if found in s), None)
            if true_sentence is None:
                continue

            _, positive_score = best_semantic_match(opinion_text, [true_sentence])
            positive_scores.append(positive_score)

            other_sentence = random.choice([s for s in all_sentences_pool if s != true_sentence])
            _, negative_score = best_semantic_match(opinion_text, [other_sentence])
            negative_scores.append(negative_score)

In [13]:
positive_series = pd.Series(positive_scores, name="par_correto")
negative_series = pd.Series(negative_scores, name="par_aleatorio")
pd.DataFrame({"correto": positive_series.describe(), "aleatorio": negative_series.describe()})

,correto,aleatorio
count,31.000000,31.000000
mean,0.502219,0.036749
std,0.192395,0.040823
min,0.049070,0.000000
25%,0.357035,0.000000
50%,0.567623,0.028742
75%,0.632790,0.061753
max,0.789768,0.174229


In [14]:
confidence_threshold = 0.25
positives_below = (positive_series < confidence_threshold).sum()
negatives_above = (negative_series >= confidence_threshold).sum()
print(f"corte escolhido: {confidence_threshold}")
print(f"pares corretos abaixo do corte: {positives_below}/{len(positive_series)}")
print(f"pares aleatórios acima do corte: {negatives_above}/{len(negative_series)}")

corte escolhido: 0.25
pares corretos abaixo do corte: 4/31
pares aleatórios acima do corte: 0/31


A mediana dos pares corretos fica em torno de 0,57, a dos pares aleatórios abaixo de 0,06: as duas distribuições se separam bem, com alguma sobreposição entre 0,05 e 0,25. Um corte de 0,25 é uma escolha razoável, não perfeita. Com esse corte, a classificação de cada opinião passa a ter quatro níveis: citação encontrada, similaridade acima do corte, similaridade abaixo do corte (uma tentativa de baixa confiança) ou pessoa não resolvida.

In [15]:
def classify_evidence(opinion_text, person_speech, sentences):
    quote = extract_quote(opinion_text)
    if quote is not None:
        found = find_quote_evidence(quote, person_speech)
        if found is not None:
            return "quote_found"

    best_sentence, score = best_semantic_match(opinion_text, sentences)
    if best_sentence is None:
        return "no_evidence"
    return "semantic_match_high" if score >= confidence_threshold else "semantic_match_weak"

In [16]:
coverage_v1 = {"quote_found": 0, "semantic_match_high": 0, "semantic_match_weak": 0, "person_not_resolved": 0}

for hearing in sample_hearings:
    turns = [dict(t, resolved_name=resolve_turn_name(t)) for t in split_into_turns(hearing["transcricao"])]
    for pessoa in hearing["metadados"]["envolvidos"]:
        matched_turns, speech = resolve_person_speech(pessoa, turns)
        sentences = split_sentences(speech)
        for opinion_text in pessoa["opinioes"]:
            if not matched_turns:
                coverage_v1["person_not_resolved"] += 1
                continue
            coverage_v1[classify_evidence(opinion_text, speech, sentences)] += 1

In [17]:
pd.DataFrame(
    {"so_citacao": pd.Series(quote_only_counts), "citacao_mais_similaridade": pd.Series(coverage_v1)}
).fillna(0)

,so_citacao,citacao_mais_similaridade
no_quote,151.0,0.0
person_not_resolved,14.0,14.0
quote_found,32.0,32.0
quote_not_found,71.0,0.0
semantic_match_high,0.0,158.0
semantic_match_weak,0.0,64.0


Das 268 opiniões, 190 (32 por citação, 158 por similaridade acima do corte) terminam com evidência de confiança razoável, e as outras 64 recebem ao menos uma tentativa de baixa confiança em vez de nada. É uma melhora real. Mas a calibração só mostra que pares já sabidamente certos têm similaridade maior do que pares aleatórios, ela não mede a taxa de acerto das evidências que a pipeline de fato produz no dia a dia, quando não há citação para conferir.

## Tentativa 3: a similaridade é confiável? Validação manual

Para medir isso de verdade, uma amostra de pares opinião-evidência dos dois níveis de confiança foi classificada manualmente num arquivo revisável (`udv_manual_review.json`), sujeito a correção: cada par recebeu `correta`, `parcial` (mesmo assunto, mas não confirma a afirmação específica) ou `incorreta`.

In [18]:
with open("udv_manual_review.json") as f:
    review_items = json.load(f)

review_df = pd.DataFrame(review_items)
print(f"itens carregados: {len(review_df)}")

itens carregados: 40


In [19]:
pd.crosstab(review_df["tier"], review_df["judgment"])

judgment,correta,incorreta,parcial
tier,,,
semantic_match_high,17,0,3
semantic_match_weak,3,9,8


In [20]:
accuracy_by_tier = review_df.groupby("tier")["judgment"].apply(lambda j: (j == "correta").mean())
accuracy_by_tier

tier
semantic_match_high    0.85
semantic_match_weak    0.15
Name: judgment, dtype: float64

Numa amostra de 20 pares por nível: `semantic_match_high` teve 17 corretos e 3 parciais, nenhum incorreto (85% de acerto). `semantic_match_weak` teve só 3 corretos, 8 parciais e 9 incorretos (15% de acerto, quase metade claramente errada). O corte de 0,25 separa bem evidência confiável de não confiável. A classificação vem de um arquivo revisável, editável à mão, sem revisão por um especialista humano independente. A amostra tem 20 pares por nível: o resultado é um indício direcional, sem intervalo de confiança formal.

## Para onde isso aponta

TF-IDF compara vocabulário compartilhado, não significado. Isso explica a taxa de erro alta na camada de baixa confiança: uma opinião que parafraseia a fala trocando palavras por sinônimos, mantendo o mesmo sentido, tende a ter similaridade baixa mesmo quando o conteúdo está correto.

O próximo passo é testar uma comparação por significado, com embeddings densos, na mesma amostra de validação manual, para ver se ela recupera os casos que caem na camada fraca por causa dessa limitação de vocabulário.